In [2]:
import torch
import numpy as np
from typing import List
from diffusers import StableDiffusionPipeline
from IPython.display import display

2026-02-18 16:20:03.558527: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771431603.741486      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771431603.792388      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771431604.235929      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771431604.235971      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771431604.235974      55 computation_placer.cc:177] computation placer alr

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    torch_dtype=torch.float16,
    safety_checker=None
).to(device)

model.enable_attention_slicing()
model.enable_vae_slicing()

print("✓ Model loaded")


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

scheduler_config-checkpoint.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


✓ Model loaded


In [4]:
@torch.no_grad()
def generate(prompt, seed=None):

    generator = None
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)

    image = model(
        prompt=prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    return image


In [6]:
# =========================================================
# RELATIONAL IMAGE GENERATION + SEED TRACKING + GT ANNOTATION
# =========================================================

from tqdm import tqdm
import torch
import gc
import os
import random
import pandas as pd
from PIL import Image
import shutil

# ------------------------
# Configuration
# ------------------------
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

total_images = 1          # number of images to generate
batch_size = 20
seed_start = 42
start_index = 0

# ------------------------
# Objects
# ------------------------
OBJECTS = [
    "cat", "dog", "horse", "car", "bicycle", "bus",
    "apple", "banana", "chair", "table",
    "bird", "boat", "airplane", "cow", "sheep",
    "tv", "laptop", "phone", "cup"
]

# ------------------------
# Relational predicates
# ------------------------
RELATIONS = [
    "slightly larger than",
    "larger than",
    "much larger than",
    "slightly smaller than",
    "smaller than",
    "taller than",
    "shorter than"
]

# =========================================================
# Generate relational prompts
# =========================================================
def generate_prompts(n):
    prompts = []
    metadata = []

    for _ in range(n):
        obj_a, obj_b = random.sample(OBJECTS, 2)
        relation = random.choice(RELATIONS)

        # Carefully worded for diffusion understanding
        prompt = (
            f" a {obj_a} is {relation} a {obj_b} "
        )

        prompts.append(prompt)

        metadata.append({
            "object_A": obj_a,
            "object_B": obj_b,
            "relation": relation
        })

    return prompts, metadata


prompts, meta_info = generate_prompts(total_images)

# ------------------------
# CSV Records
# ------------------------
records = []

# =========================================================
# Batch Generation
# =========================================================
def generate_batch(batch_prompts, batch_seeds, batch_indices):

    for i, (prompt, seed, idx) in enumerate(zip(batch_prompts, batch_seeds, batch_indices)):
        try:
            # Generate image
            img = generate(prompt=prompt, seed=seed)

            filename = f"generated_{idx:05d}.png"
            save_path = os.path.join(output_dir, filename)
            img.save(save_path)

            # Save ground-truth relation labels
            records.append({
                "image": filename,
                "prompt": prompt,
                "seed": seed,
                "object_A": meta_info[idx]["object_A"],
                "object_B": meta_info[idx]["object_B"],
                "relation": meta_info[idx]["relation"]
            })

            print(f"✓ Saved {filename}")

        except Exception as e:
            print(f"⚠️ Error at image {idx}: {e}")

        # Memory cleanup (VERY IMPORTANT for Kaggle)
        del img
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# =========================================================
# Main Loop
# =========================================================
for batch_start in tqdm(range(0, total_images, batch_size), desc="Generating images"):
    batch_end = min(batch_start + batch_size, total_images)

    batch_indices = list(range(start_index + batch_start, start_index + batch_end))
    batch_seeds = [seed_start + i for i in range(batch_start, batch_end)]
    batch_prompts = prompts[batch_start:batch_end]

    generate_batch(batch_prompts, batch_seeds, batch_indices)


# =========================================================
# Save CSV annotation (VERY IMPORTANT FOR EVALUATION)
# =========================================================
df = pd.DataFrame(records)
csv_path = os.path.join(output_dir, "prompts.csv")
df.to_csv(csv_path, index=False)

print(f"\n✅ prompts.csv saved with {len(df)} entries")
print(csv_path)


# =========================================================
# Zip output (for Kaggle download)
# =========================================================
zip_path = "output.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(base_name="output", format="zip", root_dir=output_dir)

print(f"✅ All images zipped into {zip_path}")


Generating images:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Generating images: 100%|██████████| 1/1 [00:10<00:00, 10.91s/it]

✓ Saved generated_00000.png

✅ prompts.csv saved with 1 entries
output/prompts.csv
✅ All images zipped into output.zip


In [2]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-qb1u49ir
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-qb1u49ir
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done


In [3]:
# =========================================================
# Full Evaluation Script for Generated Images
# =========================================================
import os
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# =========================================================
# CONFIG
# =========================================================
IMAGE_DIR = "/kaggle/input/datasets/mdsunzidulislam/stable-slightly"
CSV_PATH  = "/kaggle/input/datasets/mdsunzidulislam/stable-slightly/prompts.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ALPHA  = 0.3   # Predicated Diffusion weight for attribute leakage

# Threshold for binary presence/absence decision via softmax logit.
# Values above 0.5 mean "present" wins over "absent".
# You can raise this (e.g. 0.6) for stricter presence requirements.
PRESENCE_THRESHOLD = 0.5

# =========================================================
# LOAD PROMPTS
# =========================================================
df = pd.read_csv(CSV_PATH)
assert all(col in df.columns for col in ["image", "prompt", "seed"]), \
    "CSV must contain 'image', 'prompt', 'seed' columns."

# =========================================================
# SETUP CLIP MODEL
# =========================================================
print("Loading CLIP model...")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model.eval()
print(f"CLIP loaded on {DEVICE}.")

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def load_image(path: str) -> Image.Image:
    """Load an image from disk as RGB."""
    return Image.open(path).convert("RGB")


def clip_scores_batch(image: Image.Image, texts: list[str]) -> list[float]:
    """
    Compute CLIP cosine similarity between one image and multiple text prompts
    in a single forward pass for efficiency.

    Returns a list of raw cosine similarities (range ~ [-1, 1]).
    """
    inputs = clip_processor(
        text=texts,
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs = clip_model(**inputs)
        img_feat  = outputs.image_embeds                              # (1, D)
        txt_feat  = outputs.text_embeds                               # (N, D)
        img_feat  = img_feat  / img_feat.norm(p=2, dim=-1, keepdim=True)
        txt_feat  = txt_feat  / txt_feat.norm(p=2, dim=-1, keepdim=True)
        sims      = (img_feat @ txt_feat.T).squeeze(0)                # (N,)

    return sims.cpu().tolist()


def is_object_present(image: Image.Image, obj: str, threshold: float = PRESENCE_THRESHOLD) -> tuple[bool, float]:
    """
    Binary presence check using a matched positive/negative text pair.

    Instead of comparing two objects against each other (which is unreliable),
    we ask CLIP directly: "does this image contain {obj}?" by competing a
    positive caption against a negative one and using softmax to decide.

    Returns:
        (present: bool, confidence: float in [0, 1])
    """
    positive = f"a photo of a {obj}"
    negative = f"a photo without a {obj}"

    pos_score, neg_score = clip_scores_batch(image, [positive, negative])

    # Softmax over the two logits gives a calibrated probability
    scores  = torch.tensor([pos_score, neg_score])
    probs   = torch.softmax(scores * 100, dim=0)   # scale before softmax (CLIP convention)
    confidence = probs[0].item()                   # probability that object IS present

    present = confidence >= threshold
    return present, confidence


def parse_prompt(prompt: str) -> dict:
    """
    Parse size-comparison prompts of the form:
      "a [Obj A] is [size_modifier] [smaller|larger] than a [Obj B]"

    Examples:
      "a horse is slightly smaller than a cup"
      "a cat is much larger than a car"
      "a elephant is slightly smaller than a mouse"

    Returns a dict with keys:
      obj_a        : str   — the subject object  (e.g. "horse")
      obj_b        : str   — the reference object (e.g. "cup")
      size_rel     : str   — "smaller" or "larger"
      modifier     : str   — intensity word (e.g. "slightly", "much", "a lot") or ""
      obj_a_smaller: bool  — True if obj_a should appear smaller than obj_b

    Raises ValueError on unexpected format.
    """
    import re

    raw = prompt.lower().strip()

    # Pattern: "a/an <obj_a> is [modifier] <smaller|larger|bigger> than a/an <obj_b>"
    pattern = re.compile(
        r"^(?:a|an)\s+(?P<obj_a>\w+)\s+is\s+"
        r"(?P<modifier>(?:(?:a\s+lot|a\s+bit|much|slightly|far|way|somewhat|a\s+little)\s+)?)?"
        r"(?P<size_rel>smaller|larger|bigger|tinier|huger)\s+than\s+"
        r"(?:a|an)\s+(?P<obj_b>\w+)"
        r".*$"
    )

    m = pattern.match(raw)
    if not m:
        raise ValueError(
            f"Prompt does not match expected size-comparison format "
            f"'a <obj> is [modifier] smaller/larger than a <obj>'. Got: {prompt!r}"
        )

    obj_a     = m.group("obj_a").strip()
    obj_b     = m.group("obj_b").strip()
    size_rel  = m.group("size_rel").strip()
    modifier  = (m.group("modifier") or "").strip()

    # Normalise: "bigger" / "huger" → "larger", "tinier" → "smaller"
    if size_rel in ("bigger", "huger"):
        size_rel = "larger"
    if size_rel == "tinier":
        size_rel = "smaller"

    obj_a_smaller = (size_rel == "smaller")

    return {
        "obj_a":         obj_a,
        "obj_b":         obj_b,
        "size_rel":      size_rel,
        "modifier":      modifier,
        "obj_a_smaller": obj_a_smaller,
    }


# =========================================================
# EVALUATION LOOP
# =========================================================
results = []

print("Evaluating images...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    image_name = row["image"]
    prompt     = row["prompt"]
    image_path = os.path.join(IMAGE_DIR, image_name)

    if not os.path.exists(image_path):
        print(f"[WARN] Image not found, skipping: {image_path}")
        continue

    # ------------------------------------------------------------------
    # Parse prompt
    # ------------------------------------------------------------------
    try:
        parsed = parse_prompt(prompt)
    except ValueError as e:
        print(f"[WARN] Failed to parse prompt ({e}), skipping row {idx}.")
        continue

    obj_a         = parsed["obj_a"]
    obj_b         = parsed["obj_b"]
    size_rel      = parsed["size_rel"]        # "smaller" or "larger"
    modifier      = parsed["modifier"]        # e.g. "slightly", "much", ""
    obj_a_smaller = parsed["obj_a_smaller"]   # True  → obj_a should look smaller

    image = load_image(image_path)

    # ------------------------------------------------------------------
    # MISSING OBJECT DETECTION — absolute presence/absence
    # ------------------------------------------------------------------
    present_a, conf_a = is_object_present(image, obj_a)
    present_b, conf_b = is_object_present(image, obj_b)

    missing_a       = not present_a
    missing_b       = not present_b
    missing_objects = missing_a or missing_b   # strict:  at least one missing
    both_missing    = missing_a and missing_b  # lenient: both missing

    # ------------------------------------------------------------------
    # SIZE-RELATION VIOLATION DETECTION
    
    # Strategy: ask CLIP to score the image against two competing captions:
    #   caption_correct  — describes the size relation as prompted
    #   caption_violated — describes the opposite size relation
    
    # If the model painted them at normal real-world scale (e.g. horse >> cup)
    # instead of the requested counterfactual (horse < cup), the "violated"
    # caption should score higher.
    # ------------------------------------------------------------------
    opposite_rel = "larger" if obj_a_smaller else "smaller"

    caption_correct  = f"a {obj_a} that is {modifier} {size_rel} than a {obj_b}".strip()
    caption_violated = f"a {obj_a} that is {opposite_rel} than a {obj_b}".strip()

    score_correct, score_violated = clip_scores_batch(image, [caption_correct, caption_violated])

    # Softmax confidence that the correct size relation is depicted
    logits = torch.tensor([score_correct, score_violated])
    probs  = torch.softmax(logits * 100, dim=0)
    size_correct_conf = probs[0].item()          # higher → image matches the prompt

    size_violation = size_correct_conf < 0.5     # model failed to show requested scale

    # ------------------------------------------------------------------
    # Record result
    # ------------------------------------------------------------------
    results.append({
        "image":              image_name,
        "prompt":             prompt,
        # ---- parsed fields ----
        "obj_a":              obj_a,
        "obj_b":              obj_b,
        "size_rel":           size_rel,
        "modifier":           modifier,
        "obj_a_smaller":      obj_a_smaller,
        # ---- missing object ----
        "present_a":          present_a,
        "present_b":          present_b,
        "confidence_a":       round(conf_a, 4),
        "confidence_b":       round(conf_b, 4),
        "missing_obj_a":      missing_a,
        "missing_obj_b":      missing_b,
        "missing_object":     missing_objects,
        "both_missing":       both_missing,
        # ---- size relation ----
        "size_violation":     size_violation,
        "size_correct_conf":  round(size_correct_conf, 4),
        "score_correct_cap":  round(score_correct, 4),
        "score_violated_cap": round(score_violated, 4),
    })

    del image
    torch.cuda.empty_cache()

# =========================================================
# SUMMARY STATS
# =========================================================
results_df = pd.DataFrame(results)

total = len(results_df)
if total > 0:
    print(f"\n{'='*50}")
    print(f"EVALUATION SUMMARY  ({total} images)")
    print(f"{'='*50}")
    print(f"  Missing obj A:              {results_df['missing_obj_a'].sum()} / {total}  "
          f"({results_df['missing_obj_a'].mean()*100:.1f}%)")
    print(f"  Missing obj B:              {results_df['missing_obj_b'].sum()} / {total}  "
          f"({results_df['missing_obj_b'].mean()*100:.1f}%)")
    print(f"  At least 1 obj missing:     {results_df['missing_object'].sum()} / {total}  "
          f"({results_df['missing_object'].mean()*100:.1f}%)")
    print(f"  Both objs missing:          {results_df['both_missing'].sum()} / {total}  "
          f"({results_df['both_missing'].mean()*100:.1f}%)")
    print(f"  Size relation violated:     {results_df['size_violation'].sum()} / {total}  "
          f"({results_df['size_violation'].mean()*100:.1f}%)")
    print(f"  Avg size-correct confidence:{results_df['size_correct_conf'].mean():.4f}  "
          f"(>0.5 = correct scale depicted)")
    print(f"  Avg presence confidence A:  {results_df['confidence_a'].mean():.4f}")
    print(f"  Avg presence confidence B:  {results_df['confidence_b'].mean():.4f}")
    print(f"{'='*50}\n")

# =========================================================
# SAVE RESULTS
# =========================================================
results_csv = "/kaggle/working/evaluation_results.csv"
results_df.to_csv(results_csv, index=False)
print("Evaluation finished!")
print("Results saved to:", results_csv)

2026-02-20 23:20:59.887448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771629660.280685      79 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771629660.382372      79 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771629661.273827      79 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771629661.273874      79 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771629661.273877      79 computation_placer.cc:177] computation placer alr

Loading CLIP model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


...

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format 'a <obj> is [modifier] smaller/larger than a <obj>'. Got: ' a horse is shorter than a bus '), skipping row 392.


 99%|█████████▉| 397/400 [01:02<00:00,  5.15it/s]

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format 'a <obj> is [modifier] smaller/larger than a <obj>'. Got: ' a bird is taller than a sheep '), skipping row 397.


100%|██████████| 400/400 [01:02<00:00,  6.40it/s]

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format 'a <obj> is [modifier] smaller/larger than a <obj>'. Got: ' a cow is shorter than a banana '), skipping row 399.

EVALUATION SUMMARY  (285 images)
  Missing obj A:              37 / 285  (13.0%)
  Missing obj B:              127 / 285  (44.6%)
  At least 1 obj missing:     155 / 285  (54.4%)
  Both objs missing:          9 / 285  (3.2%)
  Size relation violated:     100 / 285  (35.1%)
  Avg size-correct confidence:0.6026  (>0.5 = correct scale depicted)
  Avg presence confidence A:  0.8218
  Avg presence confidence B:  0.5447

Evaluation finished!
Results saved to: /kaggle/working/evaluation_results.csv


In [4]:
# =========================================================
# Full Evaluation Script for Generated Images
# Lenient vs Strict missing-object criteria
# =========================================================
import os
import re
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# =========================================================
# CONFIG
# =========================================================
IMAGE_DIR = "/kaggle/input/datasets/mdsunzidulislam/stable-slightly"
CSV_PATH  = "/kaggle/input/datasets/mdsunzidulislam/stable-slightly/prompts.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ALPHA  = 0.3   # Predicated Diffusion weight for attribute leakage

# Threshold for binary presence/absence decision via softmax logit.
# Values above 0.5 mean "present" wins over "absent".
PRESENCE_THRESHOLD = 0.5

# =========================================================
# LOAD PROMPTS
# =========================================================
df = pd.read_csv(CSV_PATH)
assert all(col in df.columns for col in ["image", "prompt", "seed"]), \
    "CSV must contain 'image', 'prompt', 'seed' columns."

# =========================================================
# SETUP CLIP MODEL
# =========================================================
print("Loading CLIP model...")
clip_model     = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip_model.eval()
print(f"CLIP loaded on {DEVICE}.")

# =========================================================
# HELPER FUNCTIONS
# =========================================================

def load_image(path: str) -> Image.Image:
    """Load an image from disk as RGB."""
    return Image.open(path).convert("RGB")


def clip_scores_batch(image: Image.Image, texts: list[str]) -> list[float]:
    """
    Compute CLIP cosine similarity between one image and multiple text prompts
    in a single forward pass for efficiency.
    Returns a list of raw cosine similarities (range ~ [-1, 1]).
    """
    inputs = clip_processor(
        text=texts,
        images=image,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(DEVICE)

    with torch.no_grad():
        outputs  = clip_model(**inputs)
        img_feat = outputs.image_embeds
        txt_feat = outputs.text_embeds
        img_feat = img_feat / img_feat.norm(p=2, dim=-1, keepdim=True)
        txt_feat = txt_feat / txt_feat.norm(p=2, dim=-1, keepdim=True)
        sims     = (img_feat @ txt_feat.T).squeeze(0)

    return sims.cpu().tolist()


def is_object_present(
    image: Image.Image,
    obj: str,
    threshold: float = PRESENCE_THRESHOLD
) -> tuple[bool, float]:
    """
    Binary presence check using a matched positive/negative text pair.

    Returns:
        (present: bool, confidence: float in [0, 1])
    """
    positive = f"a photo of a {obj}"
    negative = f"a photo without a {obj}"

    pos_score, neg_score = clip_scores_batch(image, [positive, negative])

    scores     = torch.tensor([pos_score, neg_score])
    probs      = torch.softmax(scores * 100, dim=0)
    confidence = probs[0].item()
    present    = confidence >= threshold
    return present, confidence


def is_object_mixture(
    image: Image.Image,
    obj_a: str,
    obj_b: str,
) -> tuple[bool, float]:
    """
    Detect whether the two objects are fused/mixed into a single hybrid entity.

    Strategy: compare CLIP score of a "hybrid" caption against scores of
    clean individual-object captions.  If the hybrid caption wins, we call
    it a mixture.

    Returns:
        (is_mixture: bool, mixture_confidence: float in [0, 1])
    """
    hybrid_caption  = f"a hybrid creature that is part {obj_a} and part {obj_b}"
    clean_caption_a = f"a photo of a {obj_a} and a {obj_b} as two separate objects"

    scores = clip_scores_batch(image, [hybrid_caption, clean_caption_a])
    logits = torch.tensor(scores)
    probs  = torch.softmax(logits * 100, dim=0)

    mixture_conf = probs[0].item()     # probability that image is a hybrid
    is_mix       = mixture_conf > 0.5
    return is_mix, mixture_conf


def parse_prompt(prompt: str) -> dict:
    """
    Parse size-comparison prompts of the form:
      "a [Obj A] is [size_modifier] [smaller|larger] than a [Obj B]"

    Returns a dict with keys:
      obj_a, obj_b, size_rel, modifier, obj_a_smaller

    Raises ValueError on unexpected format.
    """
    raw = prompt.lower().strip()

    pattern = re.compile(
        r"^(?:a|an)\s+(?P<obj_a>\w+)\s+is\s+"
        r"(?P<modifier>(?:(?:a\s+lot|a\s+bit|much|slightly|far|way|somewhat|a\s+little)\s+)?)?"
        r"(?P<size_rel>smaller|larger|bigger|tinier|huger)\s+than\s+"
        r"(?:a|an)\s+(?P<obj_b>\w+)"
        r".*$"
    )

    m = pattern.match(raw)
    if not m:
        raise ValueError(
            f"Prompt does not match expected size-comparison format. Got: {prompt!r}"
        )

    obj_a    = m.group("obj_a").strip()
    obj_b    = m.group("obj_b").strip()
    size_rel = m.group("size_rel").strip()
    modifier = (m.group("modifier") or "").strip()

    if size_rel in ("bigger", "huger"):
        size_rel = "larger"
    if size_rel == "tinier":
        size_rel = "smaller"

    obj_a_smaller = (size_rel == "smaller")

    return {
        "obj_a":         obj_a,
        "obj_b":         obj_b,
        "size_rel":      size_rel,
        "modifier":      modifier,
        "obj_a_smaller": obj_a_smaller,
    }


# =========================================================
# EVALUATION LOOP
# =========================================================
results = []

print("Evaluating images...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    image_name = row["image"]
    prompt     = row["prompt"]
    image_path = os.path.join(IMAGE_DIR, image_name)

    if not os.path.exists(image_path):
        print(f"[WARN] Image not found, skipping: {image_path}")
        continue

    try:
        parsed = parse_prompt(prompt)
    except ValueError as e:
        print(f"[WARN] Failed to parse prompt ({e}), skipping row {idx}.")
        continue

    obj_a         = parsed["obj_a"]
    obj_b         = parsed["obj_b"]
    size_rel      = parsed["size_rel"]
    modifier      = parsed["modifier"]
    obj_a_smaller = parsed["obj_a_smaller"]

    image = load_image(image_path)

    # ------------------------------------------------------------------
    # STEP 1 — Raw presence detection
    # ------------------------------------------------------------------
    present_a, conf_a = is_object_present(image, obj_a)
    present_b, conf_b = is_object_present(image, obj_b)

    missing_a = not present_a
    missing_b = not present_b

    # ------------------------------------------------------------------
    # STEP 2 — Object mixture detection
    #
    # A "mixture" means both objects appear fused into one hybrid entity.
    # Paper treatment:
    #   Lenient → mixture is NOT counted as missing (model tried, just blended)
    #   Strict  → mixture IS counted as missing  (pure instances required)
    # ------------------------------------------------------------------
    is_mix, mix_conf = is_object_mixture(image, obj_a, obj_b)

    # ── LENIENT criterion ──────────────────────────────────────────
    # Object is "missing" only if it is absent AND the image is NOT
    # a mixture (i.e. we forgive blending as a form of presence).
    lenient_missing_a = missing_a and not is_mix
    lenient_missing_b = missing_b and not is_mix

    # at-least-one and both-missing under lenient
    lenient_at_least_one = lenient_missing_a or  lenient_missing_b
    lenient_both_missing = lenient_missing_a and lenient_missing_b

    # ── STRICT criterion ───────────────────────────────────────────
    # Object is "missing" if it is absent OR the image is a mixture
    # (mixture counts as failing to generate a pure instance).
    strict_missing_a = missing_a or is_mix
    strict_missing_b = missing_b or is_mix

    # at-least-one and both-missing under strict
    strict_at_least_one = strict_missing_a or  strict_missing_b
    strict_both_missing = strict_missing_a and strict_missing_b

    # ------------------------------------------------------------------
    # STEP 3 — Size-relation violation detection
    # ------------------------------------------------------------------
    opposite_rel = "larger" if obj_a_smaller else "smaller"

    caption_correct  = f"a {obj_a} that is {modifier} {size_rel} than a {obj_b}".strip()
    caption_violated = f"a {obj_a} that is {opposite_rel} than a {obj_b}".strip()

    score_correct, score_violated = clip_scores_batch(
        image, [caption_correct, caption_violated]
    )

    logits = torch.tensor([score_correct, score_violated])
    probs  = torch.softmax(logits * 100, dim=0)
    size_correct_conf = probs[0].item()
    size_violation    = size_correct_conf < 0.5

    # ------------------------------------------------------------------
    # Record
    # ------------------------------------------------------------------
    results.append({
        "image":              image_name,
        "prompt":             prompt,
        # parsed
        "obj_a":              obj_a,
        "obj_b":              obj_b,
        "size_rel":           size_rel,
        "modifier":           modifier,
        "obj_a_smaller":      obj_a_smaller,
        # raw presence
        "present_a":          present_a,
        "present_b":          present_b,
        "confidence_a":       round(conf_a, 4),
        "confidence_b":       round(conf_b, 4),
        "missing_obj_a_raw":  missing_a,
        "missing_obj_b_raw":  missing_b,
        # mixture
        "is_mixture":         is_mix,
        "mixture_confidence": round(mix_conf, 4),
        # ── LENIENT (mixture forgiven) ──────────────────────────────
        "lenient_missing_a":       lenient_missing_a,
        "lenient_missing_b":       lenient_missing_b,
        "lenient_at_least_one":    lenient_at_least_one,
        "lenient_both_missing":    lenient_both_missing,
        # ── STRICT  (mixture penalised) ─────────────────────────────
        "strict_missing_a":        strict_missing_a,
        "strict_missing_b":        strict_missing_b,
        "strict_at_least_one":     strict_at_least_one,
        "strict_both_missing":     strict_both_missing,
        # size
        "size_violation":          size_violation,
        "size_correct_conf":       round(size_correct_conf, 4),
        "score_correct_cap":       round(score_correct, 4),
        "score_violated_cap":      round(score_violated, 4),
    })

    del image
    torch.cuda.empty_cache()

# =========================================================
# SUMMARY STATS
# =========================================================
results_df = pd.DataFrame(results)
total      = len(results_df)

if total > 0:
    mix_count = results_df["is_mixture"].sum()

    print(f"\n{'='*60}")
    print(f"EVALUATION SUMMARY  ({total} images)")
    print(f"{'='*60}")

    print(f"\n  ── Object Mixture ──────────────────────────────────────")
    print(f"  Detected mixtures/hybrids:       {mix_count} / {total}  "
          f"({mix_count/total*100:.1f}%)")

    print(f"\n  ── LENIENT (mixture NOT counted as missing) ────────────")
    print(f"  Missing obj A:                   "
          f"{results_df['lenient_missing_a'].sum()} / {total}  "
          f"({results_df['lenient_missing_a'].mean()*100:.1f}%)")
    print(f"  Missing obj B:                   "
          f"{results_df['lenient_missing_b'].sum()} / {total}  "
          f"({results_df['lenient_missing_b'].mean()*100:.1f}%)")
    print(f"  At least 1 obj missing:          "
          f"{results_df['lenient_at_least_one'].sum()} / {total}  "
          f"({results_df['lenient_at_least_one'].mean()*100:.1f}%)")
    print(f"  Both objs missing:               "
          f"{results_df['lenient_both_missing'].sum()} / {total}  "
          f"({results_df['lenient_both_missing'].mean()*100:.1f}%)")

    print(f"\n  ── STRICT  (mixture IS counted as missing) ─────────────")
    print(f"  Missing obj A:                   "
          f"{results_df['strict_missing_a'].sum()} / {total}  "
          f"({results_df['strict_missing_a'].mean()*100:.1f}%)")
    print(f"  Missing obj B:                   "
          f"{results_df['strict_missing_b'].sum()} / {total}  "
          f"({results_df['strict_missing_b'].mean()*100:.1f}%)")
    print(f"  At least 1 obj missing:          "
          f"{results_df['strict_at_least_one'].sum()} / {total}  "
          f"({results_df['strict_at_least_one'].mean()*100:.1f}%)")
    print(f"  Both objs missing:               "
          f"{results_df['strict_both_missing'].sum()} / {total}  "
          f"({results_df['strict_both_missing'].mean()*100:.1f}%)")

    print(f"\n  ── Size Relation ────────────────────────────────────────")
    print(f"  Size relation violated:          "
          f"{results_df['size_violation'].sum()} / {total}  "
          f"({results_df['size_violation'].mean()*100:.1f}%)")
    print(f"  Avg size-correct confidence:     "
          f"{results_df['size_correct_conf'].mean():.4f}  "
          f"(>0.5 = correct scale depicted)")

    print(f"\n  ── Presence Confidence ──────────────────────────────────")
    print(f"  Avg presence confidence A:       "
          f"{results_df['confidence_a'].mean():.4f}")
    print(f"  Avg presence confidence B:       "
          f"{results_df['confidence_b'].mean():.4f}")
    print(f"{'='*60}\n")

# =========================================================
# SAVE RESULTS
# =========================================================
results_csv = "/kaggle/working/evaluation_results.csv"
results_df.to_csv(results_csv, index=False)
print("Evaluation finished!")
print("Results saved to:", results_csv)

Loading CLIP model...
CLIP loaded on cuda.
Evaluating images...


  0%|          | 1/400 [00:00<02:55,  2.28it/s]

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a boat is taller than a apple '), skipping row 1.
[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a cat is taller than a chair '), skipping row 2.
[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a boat is taller than a dog '), skipping row 3.


  2%|▏         | 6/400 [00:00<01:03,  6.23it/s]

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a airplane is taller than a phone '), skipping row 6.
[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a apple is shorter than a dog '), skipping row 7.


...

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a horse is shorter than a bus '), skipping row 392.


 99%|█████████▉| 397/400 [01:25<00:00,  3.62it/s]

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a bird is taller than a sheep '), skipping row 397.


100%|██████████| 400/400 [01:25<00:00,  4.67it/s]

[WARN] Failed to parse prompt (Prompt does not match expected size-comparison format. Got: ' a cow is shorter than a banana '), skipping row 399.

EVALUATION SUMMARY  (285 images)

  ── Object Mixture ──────────────────────────────────────
  Detected mixtures/hybrids:       109 / 285  (38.2%)

  ── LENIENT (mixture NOT counted as missing) ────────────
  Missing obj A:                   24 / 285  (8.4%)
  Missing obj B:                   64 / 285  (22.5%)
  At least 1 obj missing:          85 / 285  (29.8%)
  Both objs missing:               3 / 285  (1.1%)

  ── STRICT  (mixture IS counted as missing) ─────────────
  Missing obj A:                   133 / 285  (46.7%)
  Missing obj B:                   173 / 285  (60.7%)
  At least 1 obj missing:          194 / 285  (68.1%)
  Both objs missing:               112 / 285  (39.3%)

  ── Size Relation ────────────────────────────────────────
  Size relation violated:          100 / 285  (35.1%)
  Avg size-correct confidence:     0.6026  (>0